# Type 2 : 80 blinda + 5000 challenge data

In [2]:
# ============================================================
# CODE 1 — NLI MODEL TRAINING FOR MUSIC RECOMMENDATION
# ============================================================
# Trains a DeBERTa NLI model to judge whether a song matches
# a user's music request. Three labels:
#   2 = Entailment    (song MOVES TOWARD user's goal)
#   1 = Neutral       (song is unrelated)
#   0 = Contradiction (song DOES NOT MOVE TOWARD user's goal)
#
# KEY DESIGN:
#   Premise    = full conversation (user + assistant turns)
#   Hypothesis = song template + assessment label + goal
#
# This model is used in Code 2 to rerank candidate songs.
# ============================================================

# Mount Drive first — saves checkpoints if Colab disconnects
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SAVE_DIR = '/content/drive/MyDrive/music_nli_model'
import os
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'Drive checkpoint dir: {DRIVE_SAVE_DIR}')

# Install libraries
!pip install rank_bm25 sentence-transformers datasets scikit-learn transformers -q

# Imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset
import numpy as np
import random
import time
from tqdm import tqdm

print('All imports done!')

# ─────────────────────────────────────────────────────────
# LOAD DATA
# Blind-A          → highest quality, domain-matched → 3x weight
# Challenge-Dataset→ 5000 sessions for extra signal  → 1x weight
# Track-Metadata   → song info used to build hypotheses
# ─────────────────────────────────────────────────────────
t0 = time.time()

print('\nLoading Blind-A...')
blind_a        = load_dataset('talkpl-ai/TalkPlayData-Challenge-Blind-A')
blind_sessions = list(blind_a['test'])
print(f'Blind-A sessions: {len(blind_sessions)}')

print('\nLoading Challenge-Dataset (5000 sessions)...')
challenge_data     = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset')
challenge_sessions = challenge_data['train'].shuffle(seed=42).select(range(5000))
challenge_sessions = list(challenge_sessions)
print(f'Challenge sessions: {len(challenge_sessions)}')

print('\nLoading track metadata...')
track_metadata = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Metadata')
track_split    = list(track_metadata.keys())[0]
tracks_list    = list(track_metadata[track_split])

# Build track_id → track info dictionary for O(1) lookup
track_lookup = {}
for track in tracks_list:
    tid = track.get('track_id', '')
    if isinstance(tid, list): tid = tid[0] if tid else ''
    track_lookup[str(tid)] = track

print(f'Tracks loaded: {len(tracks_list)}')
print(f'Data loading took {(time.time()-t0)/60:.1f} min')

# ─────────────────────────────────────────────────────────
# TAG INDEX
# Pre-build tag sets for every track.
# Used in neutral pair generation to ensure zero tag overlap
# so neutral tracks are genuinely unrelated to positive ones.
# ─────────────────────────────────────────────────────────
def get_tag_set(track):
    tags = track.get('tag_list', [])
    if isinstance(tags, list):
        return set(str(t).strip().lower() for t in tags if str(t).strip())
    elif isinstance(tags, str) and tags.strip():
        return set(tags.strip().lower().split(','))
    return set()

track_tag_sets = {str(t.get('track_id', '')): get_tag_set(t) for t in tracks_list}
all_track_ids  = list(track_lookup.keys())

# ─────────────────────────────────────────────────────────
# TRACK TEMPLATE
# Converts raw song metadata into a natural English sentence.
# Example: "Someone Like You by Adele from the album 21
#           is emotional, melancholic, heartbreak, piano."
# This forms the first part of every NLI hypothesis.
# ─────────────────────────────────────────────────────────
def build_track_template(track):
    track_name  = str(track.get('track_name',  'Unknown Track'))
    artist_name = str(track.get('artist_name', 'Unknown Artist'))
    album_name  = str(track.get('album_name',  'Unknown Album'))
    tags = track.get('tag_list', [])
    if isinstance(tags, list):
        tag_str = ', '.join([str(t).strip() for t in tags if str(t).strip()])
    elif isinstance(tags, str) and tags.strip():
        tag_str = tags.strip()
    else:
        tag_str = 'general'
    return f"{track_name} by {artist_name} from the album {album_name} is {tag_str}."

# ─────────────────────────────────────────────────────────
# EXTRACT REAL TRAINING PAIRS (FIXED VERSION)
#
# PREMISE: full conversation including both user AND assistant
#   turns (last 6 turns). Assistant turns carry context about
#   what was already suggested and what the user reacted to.
#
# HYPOTHESIS: song template + assessment label text + goal.
#   The assessment label is now INSIDE the hypothesis text:
#     label=2 → "moves toward the user's goal of ..."
#     label=0 → "does not move toward the user's goal of ..."
#   This gives the model a direct text signal to learn from,
#   instead of just a number label with identical hypotheses.
#
# Example pair (label=2):
#   Premise:    "Conversation: USER: I feel sad | ASSISTANT:
#                I'll find something emotional | USER: Yes slow please"
#   Hypothesis: "Someone Like You by Adele from the album 21
#                is emotional, melancholic — this song moves
#                toward the user's goal of finding comfort music."
#
# Example pair (label=0):
#   Premise:    "Conversation: USER: I feel sad | ASSISTANT:
#                I'll find something emotional | USER: Yes slow please"
#   Hypothesis: "Levels by Avicii from the album True is energetic,
#                party, dance — this song does not move toward
#                the user's goal of finding comfort music."
# ─────────────────────────────────────────────────────────
def extract_training_pairs(session):
    pairs         = []
    conversations = session.get('conversations', [])
    goal          = session.get('conversation_goal', {})
    assessments   = session.get('goal_progress_assessments', [])
    listener_goal = str(goal.get('listener_goal', ''))

    # Map turn_number → assessment label for quick lookup
    assessment_by_turn = {}
    for a in assessments:
        turn  = a.get('turn_number')
        label = a.get('goal_progress_assessment')
        if turn is not None and label is not None:
            assessment_by_turn[turn] = label

    for i, turn in enumerate(conversations):
        if turn.get('role') != 'music':
            continue

        turn_number = turn.get('turn_number')
        track_id    = str(turn.get('content', ''))

        if turn_number not in assessment_by_turn:
            continue

        label_str = assessment_by_turn[turn_number]
        if label_str == 'MOVES_TOWARD_GOAL':
            label = 2
        elif label_str == 'DOES_NOT_MOVE_TOWARD_GOAL':
            label = 0
        else:
            continue

        if track_id not in track_lookup:
            continue

        # PREMISE: last 6 turns of full conversation
        # (3 user + 3 assistant) before this music recommendation
        prior_turns = conversations[:i]
        conv_turns  = [t for t in prior_turns
                       if t.get('role') in ['user', 'assistant']]
        conv_text   = ' | '.join([
            f"{t.get('role', '').upper()}: {str(t.get('content', ''))}"
            for t in conv_turns[-6:]
        ])
        if not conv_text.strip():
            continue
        premise = f"Conversation: {conv_text}"

        # HYPOTHESIS: song template + assessment label text + goal
        # Assessment label is embedded in the hypothesis text itself
        track      = track_lookup[track_id]
        template   = build_track_template(track).rstrip('.')
        assessment = "moves toward" if label == 2 else "does not move toward"
        hypothesis = f"{template} — this song {assessment} the user's goal of {listener_goal}."

        pairs.append({'premise': premise, 'hypothesis': hypothesis, 'label': label})

    return pairs

# ─────────────────────────────────────────────────────────
# NEUTRAL PAIRS (improved)
# Only use tracks with ZERO tag overlap with the positive
# track — guarantees the neutral song is genuinely unrelated.
# Premise format matches extract_training_pairs (full conversation).
# ─────────────────────────────────────────────────────────
def build_neutral_pairs(sessions, n=400):
    neutral_pairs = []
    sampled = random.sample(sessions, min(n, len(sessions)))

    for session in sampled:
        conversations = session.get('conversations', [])
        assessments   = session.get('goal_progress_assessments', [])
        goal          = session.get('conversation_goal', {})
        listener_goal = str(goal.get('listener_goal', ''))

        # Get tag set of the positive track for this session
        pos_tags = set()
        for a in assessments:
            if a.get('goal_progress_assessment') == 'MOVES_TOWARD_GOAL':
                for turn in conversations:
                    if (turn.get('role') == 'music' and
                            turn.get('turn_number') == a.get('turn_number')):
                        tid      = str(turn.get('content', ''))
                        pos_tags = track_tag_sets.get(tid, set())
                        break
                if pos_tags:
                    break

        # Build premise from full conversation (user + assistant)
        conv_turns = [t for t in conversations
                      if t.get('role') in ['user', 'assistant']]
        conv_text  = ' | '.join([
            f"{t.get('role', '').upper()}: {str(t.get('content', ''))}"
            for t in conv_turns[-6:]
        ])
        if not conv_text.strip():
            continue
        premise = f"Conversation: {conv_text}"

        # Find a track with zero tag overlap (up to 20 attempts)
        chosen_track = None
        for _ in range(20):
            candidate_id = random.choice(all_track_ids)
            cand_tags    = track_tag_sets.get(candidate_id, set())
            if len(pos_tags & cand_tags) == 0:
                chosen_track = track_lookup[candidate_id]
                break
        if chosen_track is None:
            chosen_track = track_lookup[random.choice(all_track_ids)]

        # Neutral hypothesis — use "moves toward" since label=1 means unrelated
        template   = build_track_template(chosen_track).rstrip('.')
        hypothesis = f"{template} — this song moves toward the user's goal of {listener_goal}."
        neutral_pairs.append({'premise': premise, 'hypothesis': hypothesis, 'label': 1})

    return neutral_pairs

# ─────────────────────────────────────────────────────────
# HARD NEGATIVES
# Takes a real conversation but pairs it with a DIFFERENT
# session's goal → model learns to detect goal mismatch.
# Premise format matches extract_training_pairs.
# ─────────────────────────────────────────────────────────
def build_hard_negatives(sessions, n=300):
    hard_neg_pairs = []
    valid_sessions = []
    for s in sessions:
        goal = s.get('conversation_goal', {})
        lg   = str(goal.get('listener_goal', '')) if goal else ''
        conv_turns = [t for t in s.get('conversations', [])
                      if t.get('role') in ['user', 'assistant']]
        if lg and conv_turns:
            valid_sessions.append((s, lg, conv_turns))

    if not valid_sessions:
        return hard_neg_pairs

    sampled = random.sample(valid_sessions, min(n, len(valid_sessions)))
    goals   = [lg for _, lg, _ in valid_sessions]

    for session, own_goal, conv_turns in sampled:
        # Build premise from full conversation (user + assistant)
        conv_text = ' | '.join([
            f"{t.get('role', '').upper()}: {str(t.get('content', ''))}"
            for t in conv_turns[-6:]
        ])
        if not conv_text.strip():
            continue
        premise = f"Conversation: {conv_text}"

        # Use a different session's goal → guaranteed mismatch
        other_goals = [g for g in goals if g != own_goal]
        other_goal  = random.choice(other_goals if other_goals else goals)

        # "does not move toward" + wrong goal = clear contradiction signal
        track      = track_lookup[random.choice(all_track_ids)]
        template   = build_track_template(track).rstrip('.')
        hypothesis = f"{template} — this song does not move toward the user's goal of {other_goal}."
        hard_neg_pairs.append({'premise': premise, 'hypothesis': hypothesis, 'label': 0})

    return hard_neg_pairs

# ─────────────────────────────────────────────────────────
# BUILD COMPLETE TRAINING SET
# Blind-A × 3 + Challenge × 1 + neutral pairs + hard negatives
# Shuffled to prevent model learning order patterns
# ─────────────────────────────────────────────────────────
t1 = time.time()

print('\nExtracting pairs from Blind-A...')
blind_pairs = []
for session in tqdm(blind_sessions, desc='Blind-A'):
    blind_pairs.extend(extract_training_pairs(session))
print(f'Blind-A pairs: {len(blind_pairs)}')

print('\nExtracting pairs from Challenge-Dataset (5000 sessions)...')
challenge_pairs = []
for session in tqdm(challenge_sessions, desc='Challenge'):
    challenge_pairs.extend(extract_training_pairs(session))
print(f'Challenge pairs: {len(challenge_pairs)}')

# Blind-A weighted 3x — domain-matched to test set (Blind-B)
training_pairs = (blind_pairs * 3) + challenge_pairs

print('\nBuilding neutral pairs (tag-overlap filtered)...')
all_sessions  = blind_sessions + challenge_sessions
neutral_pairs = build_neutral_pairs(all_sessions, n=400)
print(f'Neutral pairs: {len(neutral_pairs)}')

print('Building hard negatives (goal-mismatch)...')
hard_neg_pairs = build_hard_negatives(challenge_sessions, n=300)
print(f'Hard negative pairs: {len(hard_neg_pairs)}')

training_pairs = training_pairs + neutral_pairs + hard_neg_pairs
random.shuffle(training_pairs)

print(f'\n=== FINAL TRAINING SET ===')
print(f'Total pairs:     {len(training_pairs)}')
print(f'  Entailment:    {sum(1 for p in training_pairs if p["label"]==2)}')
print(f'  Neutral:       {sum(1 for p in training_pairs if p["label"]==1)}')
print(f'  Contradiction: {sum(1 for p in training_pairs if p["label"]==0)}')
print(f'Pair building took {(time.time()-t1)/60:.1f} min')

# ─────────────────────────────────────────────────────────
# PYTORCH DATASET
# Tokenizes premise + hypothesis together as one input.
# Format: [CLS] premise [SEP] hypothesis [SEP]
# max_length=256 balances context vs T4 memory usage.
# ─────────────────────────────────────────────────────────
class NLIMusicDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=256):
        self.pairs     = pairs
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair    = self.pairs[idx]
        encoded = self.tokenizer(
            pair['premise'], pair['hypothesis'],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'label':          torch.tensor(pair['label'], dtype=torch.long)
        }

# ─────────────────────────────────────────────────────────
# LOAD DEBERTA MODEL
# nli-deberta-v3-small: pre-trained on NLI, fine-tuned here
# on music domain. 'small' chosen for T4 VRAM safety.
# batch_size=16 — DeBERTa is memory-hungry, 32 risks OOM.
# ─────────────────────────────────────────────────────────
print('\nLoading NLI model...')
MODEL_NAME = 'cross-encoder/nli-deberta-v3-small'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model      = model.to(device)
print(f'Model on {device}')

split         = int(0.9 * len(training_pairs))
train_dataset = NLIMusicDataset(training_pairs[:split], tokenizer)
val_dataset   = NLIMusicDataset(training_pairs[split:], tokenizer)
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# ─────────────────────────────────────────────────────────
# TRAINING — 2 EPOCHS
# AdamW + linear warmup — standard for transformer fine-tuning.
# Gradient clipping (max 1.0) prevents exploding gradients.
# Checkpoint saved to Drive after every epoch (crash safety).
# Best val_acc model saved separately for use in Code 2.
# ─────────────────────────────────────────────────────────
EPOCHS      = 2
optimizer   = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, total_steps // 10),
    num_training_steps=total_steps
)
loss_fn      = nn.CrossEntropyLoss()
best_val_acc = 0.0

print(f'\nTraining: {len(train_dataset)} train | {len(val_dataset)} val')
print(f'Steps per epoch: {len(train_loader)} | Total steps: {total_steps}')
print(f'Estimated time: ~{len(train_loader) * EPOCHS * 0.35 / 60:.0f} min on T4\n')

t2 = time.time()
for epoch in range(EPOCHS):

    # Training phase
    model.train()
    total_loss, correct, total = 0, 0, 0
    epoch_start = time.time()

    for i, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs    = model(input_ids=input_ids, attention_mask=attention_mask)
        loss       = loss_fn(outputs.logits, labels)
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        preds   = torch.argmax(outputs.logits, dim=-1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

        if i % 50 == 0:
            elapsed = (time.time() - epoch_start) / 60
            print(f'  Epoch {epoch+1} | Step {i}/{len(train_loader)} '
                  f'| Loss: {loss.item():.4f} | Elapsed: {elapsed:.1f}min')

    train_acc = correct / total

    # Validation phase
    model.eval()
    vc, vt, val_loss_total = 0, 0, 0.0
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            val_loss_total += loss_fn(outputs.logits, labels).item()
            preds  = torch.argmax(outputs.logits, dim=-1)
            vc    += (preds == labels).sum().item()
            vt    += labels.size(0)

    val_acc  = vc / vt
    val_loss = val_loss_total / len(val_loader)
    print(f'\n✅ Epoch {epoch+1} done in {(time.time()-epoch_start)/60:.1f}min')
    print(f'   train_acc={train_acc:.3f} | val_acc={val_acc:.3f} | val_loss={val_loss:.4f}')

    # Save epoch checkpoint to Drive (crash protection)
    epoch_dir = os.path.join(DRIVE_SAVE_DIR, f'epoch_{epoch+1}')
    os.makedirs(epoch_dir, exist_ok=True)
    model.save_pretrained(epoch_dir)
    tokenizer.save_pretrained(epoch_dir)
    print(f'   📁 Checkpoint saved: {epoch_dir}')

    # Save best model — this is what Code 2 loads for inference
    # Save best model — this is what Code 2 loads for inference
if val_acc > best_val_acc:
    best_val_acc = val_acc
    best_dir = os.path.join(DRIVE_SAVE_DIR, 'best')
    os.makedirs(best_dir, exist_ok=True)
    model.save_pretrained(best_dir)
    tokenizer.save_pretrained(best_dir)
    print(f'🏆 Best val_acc={val_acc:.3f} ➜ saved to {best_dir}')

print()

print(f'Total training time: {(time.time()-t2)/60:.1f} min')
print(f'Best val_acc: {best_val_acc:.3f}')

# Save final model locally + to Drive
model.save_pretrained('./music_nli_model')
tokenizer.save_pretrained('./music_nli_model')
print('\n✅ Saved locally: ./music_nli_model')

final_drive = os.path.join(DRIVE_SAVE_DIR, 'final')
os.makedirs(final_drive, exist_ok=True)
model.save_pretrained(final_drive)
tokenizer.save_pretrained(final_drive)
print(f'✅ Saved to Drive: {final_drive}')

print('\n=== TRAINING COMPLETE ===')
print(f'Load best model in Code 2 from: {DRIVE_SAVE_DIR}/best')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive checkpoint dir: /content/drive/MyDrive/music_nli_model
All imports done!

Loading Blind-A...
Blind-A sessions: 80

Loading Challenge-Dataset (5000 sessions)...
Challenge sessions: 5000

Loading track metadata...
Tracks loaded: 47071
Data loading took 0.3 min

Extracting pairs from Blind-A...


Blind-A: 100%|██████████| 80/80 [00:00<00:00, 29233.69it/s]


Blind-A pairs: 150

Extracting pairs from Challenge-Dataset (5000 sessions)...


Challenge: 100%|██████████| 5000/5000 [00:00<00:00, 8384.37it/s]


Challenge pairs: 35000

Building neutral pairs (tag-overlap filtered)...
Neutral pairs: 400
Building hard negatives (goal-mismatch)...
Hard negative pairs: 300

=== FINAL TRAINING SET ===
Total pairs:     36150
  Entailment:    17968
  Neutral:       400
  Contradiction: 17782
Pair building took 0.0 min

Loading NLI model...


...

   📁 Checkpoint saved: /content/drive/MyDrive/music_nli_model/epoch_1
  Epoch 2 | Step 0/2034 | Loss: 0.0714 | Elapsed: 0.0min
  Epoch 2 | Step 50/2034 | Loss: 0.4493 | Elapsed: 0.4min
  Epoch 2 | Step 100/2034 | Loss: 0.1982 | Elapsed: 0.9min
  Epoch 2 | Step 150/2034 | Loss: 0.1010 | Elapsed: 1.3min
  Epoch 2 | Step 200/2034 | Loss: 0.0932 | Elapsed: 1.7min
  Epoch 2 | Step 250/2034 | Loss: 0.3317 | Elapsed: 2.2min
  Epoch 2 | Step 300/2034 | Loss: 0.2994 | Elapsed: 2.6min
  Epoch 2 | Step 350/2034 | Loss: 0.1231 | Elapsed: 3.0min
  Epoch 2 | Step 400/2034 | Loss: 0.2383 | Elapsed: 3.5min
  Epoch 2 | Step 450/2034 | Loss: 0.2942 | Elapsed: 3.9min
  Epoch 2 | Step 500/2034 | Loss: 0.1011 | Elapsed: 4.3min
  Epoch 2 | Step 550/2034 | Loss: 0.2137 | Elapsed: 4.8min
  Epoch 2 | Step 600/2034 | Loss: 0.0471 | Elapsed: 5.2min
  Epoch 2 | Step 650/2034 | Loss: 0.2106 | Elapsed: 5.6min
  Epoch 2 | Step 700/2034 | Loss: 0.3140 | Elapsed: 6.1min
  Epoch 2 | Step 750/2034 | Loss: 0.2044 | Elaps

   📁 Checkpoint saved: /content/drive/MyDrive/music_nli_model/epoch_2


🏆 Best val_acc=0.868 ➜ saved to /content/drive/MyDrive/music_nli_model/best

Total training time: 36.5 min
Best val_acc: 0.868



✅ Saved locally: ./music_nli_model


✅ Saved to Drive: /content/drive/MyDrive/music_nli_model/final

=== TRAINING COMPLETE ===
Load best model in Code 2 from: /content/drive/MyDrive/music_nli_model/best


# Code 2 : Fixed version

In [3]:
# ============================================================
# CODE 2 — MUSIC RECOMMENDATION INFERENCE (FIXED VERSION)
# ============================================================
# Uses the trained NLI model from Code 1 to rerank candidate
# songs for each Blind-B session and generate responses.
#
# Pipeline:
#   BM25 + Semantic Search → top 50
#   Cross-encoder reranking → top 30
#   NLI reranking (pure score) → top 25
#   LLM response generation
#
# KEY FIXES vs original:
#   1. Premise format matches Code 1 training (full conversation)
#   2. Hypothesis uses "moves toward" (matches training)
#   3. Pure NLI score — no circular position_score
#   4. 25 NLI candidates (was 15)
#   5. Fixed cold start query (no triple repeat)
#   6. No silent except — errors are logged
#   7. No session skipping — fallback for empty user turns
# ============================================================

# Install libraries
!pip install rank_bm25 sentence-transformers datasets scikit-learn groq transformers -q

# Mount Google Drive to access saved models
from google.colab import drive
drive.mount('/content/drive')

import torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from groq import Groq
import numpy as np, json, zipfile, time
from tqdm import tqdm

# ─────────────────────────────────────────────────────────
# API SETUP
# Store keys as variables — never hardcode in shared code
# ─────────────────────────────────────────────────────────
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client       = Groq(api_key=GROQ_API_KEY)

# ─────────────────────────────────────────────────────────
# LOAD BLIND-B SESSIONS
# These are the 80 test sessions we need to predict for
# ─────────────────────────────────────────────────────────
print('Loading Blind-B...')
blind_b             = load_dataset('talkpl-ai/TalkPlayData-Challenge-Blind-B')
blind_split         = list(blind_b.keys())[0]
blind_conversations = list(blind_b[blind_split])
print(f'Loaded {len(blind_conversations)} Blind-B sessions')

cold = sum(1 for s in blind_conversations
           if not s.get('user_profile') or s['user_profile'].get('age') is None)
print(f'Cold start sessions: {cold}')
print(f'Normal sessions:     {len(blind_conversations)-cold}')

# ─────────────────────────────────────────────────────────
# LOAD TRACK METADATA
# ─────────────────────────────────────────────────────────
print('Loading track metadata...')
track_metadata = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Metadata')
track_split    = list(track_metadata.keys())[0]
tracks_list    = list(track_metadata[track_split])
print(f'Loaded {len(tracks_list)} songs')

# ─────────────────────────────────────────────────────────
# TRACK TEMPLATE
# Converts song metadata into natural English sentence.
# Must match Code 1 exactly — same function, same output.
# ─────────────────────────────────────────────────────────
def build_track_template(track):
    track_name  = str(track.get('track_name',  'Unknown Track'))
    artist_name = str(track.get('artist_name', 'Unknown Artist'))
    album_name  = str(track.get('album_name',  'Unknown Album'))
    tags = track.get('tag_list', [])
    if isinstance(tags, list):
        tag_str = ', '.join([str(t).strip() for t in tags if str(t).strip()])
    elif isinstance(tags, str) and tags.strip():
        tag_str = tags.strip()
    else:
        tag_str = 'general'
    return f"{track_name} by {artist_name} from the album {album_name} is {tag_str}."

print('Building track templates...')
track_texts = [build_track_template(t) for t in tracks_list]

# ─────────────────────────────────────────────────────────
# BM25 INDEX (sparse retrieval)
# Keyword-based search — fast first-stage retrieval
# ─────────────────────────────────────────────────────────
bm25 = BM25Okapi([t.lower().split() for t in track_texts])
print('BM25 ready')

# ─────────────────────────────────────────────────────────
# BGE EMBEDDINGS (dense retrieval)
# Semantic search — captures meaning beyond keywords
# ─────────────────────────────────────────────────────────
embedder = SentenceTransformer('BAAI/bge-small-en-v1.5')
print('Building song vectors...')
track_embeddings = embedder.encode(
    track_texts, batch_size=256, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
print('Song vectors ready')
torch.cuda.empty_cache()

# ─────────────────────────────────────────────────────────
# CROSS-ENCODER (second-stage reranker)
# More powerful than BM25/semantic but slower — used on
# top 50 only to reduce computation
# ─────────────────────────────────────────────────────────
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Cross-encoder ready')

# ─────────────────────────────────────────────────────────
# LOAD TRAINED NLI MODEL FROM DRIVE
# Load from best checkpoint saved by Code 1.
# If Colab session still alive use local path instead:
#   './music_nli_model'
# ─────────────────────────────────────────────────────────
print('Loading trained NLI model...')
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
nli_tokenizer = AutoTokenizer.from_pretrained(
    '/content/drive/MyDrive/music_nli_model/best')
nli_model     = AutoModelForSequenceClassification.from_pretrained(
    '/content/drive/MyDrive/music_nli_model/best')
nli_model.eval()
nli_model = nli_model.to(device)
print(f'NLI model ready on {device}')

# ─────────────────────────────────────────────────────────
# RRF FUSION
# Combines BM25 and semantic rankings into one score.
# Formula: 1/(k + bm25_rank) + 1/(k + semantic_rank)
# k=60 is standard — prevents top ranks dominating too much
# ─────────────────────────────────────────────────────────
def rrf_combine(bm25_scores, sem_scores, k=60):
    bm25_ranks = np.argsort(np.argsort(-bm25_scores))
    sem_ranks  = np.argsort(np.argsort(-sem_scores))
    return 1.0 / (k + bm25_ranks + 1) + 1.0 / (k + sem_ranks + 1)

# ─────────────────────────────────────────────────────────
# NLI INFERENCE
# Runs the trained DeBERTa model on one premise+hypothesis pair.
# Returns: (entailment_prob, contradiction_prob, neutral_prob)
# Errors are logged instead of silently swallowed.
# ─────────────────────────────────────────────────────────
def nli_infer_trained(premise, hypothesis):
    try:
        inputs = nli_tokenizer(
            premise, hypothesis, return_tensors='pt',
            truncation=True, max_length=256, padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = nli_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        return float(probs[2]), float(probs[0]), float(probs[1])
    except Exception as e:
        print(f'NLI error: {e}')  # log error instead of silent fail
        return 0.0, 0.0, 1.0

# ─────────────────────────────────────────────────────────
# BUILD HYPOTHESIS (FIXED)
# Must match Code 1 training format exactly.
# Uses "moves toward" — same phrase used in training pairs.
# ─────────────────────────────────────────────────────────
def build_hypothesis(track, listener_goal):
    template = build_track_template(track).rstrip('.')
    if listener_goal:
        return f"{template} — this song moves toward the user's goal of {listener_goal}."
    else:
        return f"{template} — this song moves toward the user's goal of finding good music."

# ─────────────────────────────────────────────────────────
# NLI RERANKING (FIXED)
# Pure NLI score — position_score removed.
# Old version averaged NLI with cross-encoder position which
# cancelled out the NLI signal. Now NLI ranks independently.
# Score = entailment - 0.5 * contradiction
#   High entailment → song matches goal → high score
#   High contradiction → song conflicts → penalized
# ─────────────────────────────────────────────────────────
def nli_order_tracks(candidate_tracks, candidate_ids, premise, listener_goal):
    scored = []
    for track, tid in zip(candidate_tracks, candidate_ids):
        hypothesis             = build_hypothesis(track, listener_goal)
        entailment, contradiction, neutral = nli_infer_trained(premise, hypothesis)
        nli_score              = entailment - (0.5 * contradiction)
        scored.append({'track': track, 'tid': tid, 'final_score': nli_score})
    scored.sort(key=lambda x: x['final_score'], reverse=True)
    return scored

# ─────────────────────────────────────────────────────────
# EMOTION DETECTION
# Simple keyword-based mood detector from conversation text.
# Used to boost retrieval query with relevant mood terms.
# ─────────────────────────────────────────────────────────
def detect_emotion(text):
    text = text.lower()
    if any(w in text for w in ['sad','crying','depressed','heartbreak','lonely','grief','miss']):
        return 'sad'
    elif any(w in text for w in ['angry','mad','furious','frustrated','annoyed']):
        return 'angry'
    elif any(w in text for w in ['gym','workout','motivat','pump','training','run','exercise']):
        return 'motivational'
    elif any(w in text for w in ['study','focus','concentrat','homework','exam']):
        return 'focus'
    elif any(w in text for w in ['nostalgic','throwback','childhood','memories']):
        return 'nostalgic'
    elif any(w in text for w in ['romantic','love','date','crush','wedding']):
        return 'romantic'
    elif any(w in text for w in ['happy','excited','party','dance','energy','fun']):
        return 'happy'
    elif any(w in text for w in ['stressed','anxious','tired','relax','calm','sleep']):
        return 'calm'
    return 'neutral'

emotion_tags = {
    'sad':          'emotional melancholic sad slow ballad',
    'angry':        'aggressive intense rock metal heavy',
    'motivational': 'energetic motivational workout pump gym',
    'focus':        'instrumental lo-fi ambient focus concentration',
    'nostalgic':    'throwback classic retro old songs',
    'romantic':     'romantic love soft gentle sweet',
    'happy':        'upbeat happy energetic dance pop fun',
    'calm':         'calm relaxing peaceful ambient soft',
    'neutral':      'popular mainstream'
}

# ─────────────────────────────────────────────────────────
# LLM RESPONSE GENERATION
# Uses Groq LLaMA to generate a natural 2-sentence response
# mentioning the top recommended tracks by name.
# ─────────────────────────────────────────────────────────
def get_llm_response(conversations, top_tracks, listener_goal,
                     music_culture, country, age_group, is_cold_start):
    conv_text = ''
    for t in conversations[-4:]:
        if t.get('role') in ['user', 'assistant']:
            role = 'User' if t.get('role') == 'user' else 'Assistant'
            conv_text += f"{role}: {t.get('content','')}\n"

    tracks_text = '\n'.join([
        f"{i+1}. '{t.get('track_name','')}' by {t.get('artist_name','')}"
        for i, t in enumerate(top_tracks[:3])
    ])
    context   = "a music listener" if is_cold_start else \
                f"a {age_group} listener from {country} who prefers {music_culture}"
    goal_text = f"Goal: {listener_goal}" if listener_goal else \
                "Goal: find music that matches the conversation"

    prompt = f"""You are a friendly music assistant helping {context}.
Conversation:
{conv_text}
{goal_text}
Recommended (NLI-ranked, most suitable first):
{tracks_text}
Write exactly 2 warm sentences recommending these tracks. Mention track names naturally."""

    try:
        r = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=120, temperature=0.6
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        print(f'LLM error: {e}')
        return "Based on your conversation, I recommend these tracks for you!"

# ─────────────────────────────────────────────────────────
# MAIN PREDICTION LOOP — 80 Blind-B sessions
# ─────────────────────────────────────────────────────────
all_predictions = []

for session in tqdm(blind_conversations):
    session_id    = session['session_id']
    conversations = session.get('conversations', [])
    user_id       = session.get('user_id', '') or session_id
    profile       = session.get('user_profile', {}) or {}
    music_culture = str(profile.get('preferred_musical_culture', '') or '')
    country       = str(profile.get('country_name', '') or '')
    age_group     = str(profile.get('age_group', '') or '')
    is_cold_start = not profile or profile.get('age') is None
    goal          = session.get('conversation_goal') or {}
    listener_goal = str(goal.get('listener_goal', '') if goal else '')

    user_turns = [t for t in conversations if t.get('role') == 'user']

    # FIX: no more skipping sessions with no user turns
    # Fallback prediction prevents missing sessions in submission
    if not user_turns:
        all_predictions.append({
            'session_id':          session_id,
            'user_id':             user_id,
            'turn_number':         1,
            'predicted_track_ids': [str(tracks_list[i].get('track_id', ''))
                                    for i in range(20)],
            'predicted_response':  'Here are some songs for you!'
        })
        continue

    turn_number = user_turns[-1].get('turn_number', 1)

    # Detect emotion from full conversation text
    full_text     = ' '.join([str(t.get('content', '')) for t in conversations
                              if t.get('role') in ['user', 'assistant']])
    emotion       = detect_emotion(full_text)
    emotion_boost = emotion_tags.get(emotion, '')

    # FIX: Premise built from full conversation (user + assistant)
    # Matches Code 1 training format exactly
    conv_turns  = [t for t in conversations
                   if t.get('role') in ['user', 'assistant']]
    conv_text   = ' | '.join([
        f"{t.get('role', '').upper()}: {str(t.get('content', ''))}"
        for t in conv_turns[-6:]
    ])
    nli_premise = f"Conversation: {conv_text}"

    # Recent user text for retrieval query
    recent_text = ' '.join([str(t.get('content', '')) for t in user_turns[-3:]])

    # FIX: Cold start query — clean, no triple repeat
    if is_cold_start:
        query = f"{recent_text} {emotion_boost}".strip()
    else:
        query = f"{recent_text} {listener_goal} {music_culture} {country} {age_group} {emotion_boost}".strip()

    # ── STAGE 1: BM25 + Semantic → top 50 ────────────────
    bm25_scores = bm25.get_scores(query.lower().split())
    q_emb       = embedder.encode([query], convert_to_numpy=True,
                                   normalize_embeddings=True)
    sem_scores  = track_embeddings @ q_emb[0]
    combined    = rrf_combine(bm25_scores, sem_scores)
    top_idx     = combined.argsort()[::-1][:50]

    # ── STAGE 2: Cross-encoder → top 30 ──────────────────
    pairs          = [[query, track_texts[idx]] for idx in top_idx]
    rerank_scores  = reranker.predict(pairs)
    reranked_order = top_idx[np.argsort(rerank_scores)[::-1]][:30]

    # ── BUILD CANDIDATES: 25 NLI + 5 fill ────────────────
    # Increased from 15 → 25 so NLI sees more candidates
    # Diversity enforced: one track per artist
    nli_cands, fill_cands = [], []
    nli_ids,   fill_ids   = [], []
    seen_tracks, seen_artists = set(), set()

    for idx in reranked_order:
        track  = tracks_list[idx]
        tid    = track.get('track_id', '')
        if isinstance(tid, list): tid = tid[0] if tid else ''
        tid    = str(tid)
        artist = str(track.get('artist_name', ''))

        if tid and tid not in seen_tracks and artist not in seen_artists:
            if len(nli_cands) < 25:
                nli_cands.append(track)
                nli_ids.append(tid)
            else:
                fill_cands.append(track)
                fill_ids.append(tid)
            seen_tracks.add(tid)
            seen_artists.add(artist)

        if len(nli_cands) + len(fill_cands) >= 30:
            break

    # Fill remaining slots if diversity filter left gaps
    for idx in reranked_order:
        if len(nli_cands) + len(fill_cands) >= 30:
            break
        tid = str(tracks_list[idx].get('track_id', ''))
        if tid and tid not in seen_tracks:
            fill_cands.append(tracks_list[idx])
            fill_ids.append(tid)
            seen_tracks.add(tid)

    # ── STAGE 3: Pure NLI reranking ───────────────────────
    scored = nli_order_tracks(nli_cands, nli_ids, nli_premise, listener_goal)

    final_track_ids = [str(s['tid'])   for s in scored] + fill_ids
    final_tracks    = [s['track'] for s in scored] + fill_cands
    # Ensure exactly 20 predicted tracks
    final_track_ids = final_track_ids[:20]
    final_tracks = final_tracks[:20]

    # ── GENERATE RESPONSE ─────────────────────────────────
    response = get_llm_response(
        conversations, final_tracks, listener_goal,
        music_culture, country, age_group, is_cold_start
    )
    time.sleep(0.8)  # Groq rate limit protection

    all_predictions.append({
        'session_id':          session_id,
        'user_id':             user_id,
        'turn_number':         turn_number,
        'predicted_track_ids': final_track_ids,
        'predicted_response':  response
    })

# ─────────────────────────────────────────────────────────
# SAVE AND SUBMIT
# Must be exactly 80 predictions in prediction.json
# inside submission.zip — challenge requirement
# ─────────────────────────────────────────────────────────
print(f'\nDone! {len(all_predictions)} predictions made')
print(f'Expected 80 — Got {len(all_predictions)}')

if len(all_predictions) == 80:
    print('All 80 predictions ready — safe to submit')

    # Ensure all track IDs are strings
    for pred in all_predictions:
        pred['predicted_track_ids'] = [str(tid) for tid in pred['predicted_track_ids']]

    with open('prediction.json', 'w') as f:
        json.dump(all_predictions, f, indent=2)

    with zipfile.ZipFile('submission.zip', 'w') as z:
        z.write('prediction.json')

    print('submission.zip ready!')
    from google.colab import files
    files.download('submission.zip')

else:
    print(f'WARNING: Expected 80 but got {len(all_predictions)} — DO NOT SUBMIT')
    print('Check which sessions failed and fix before submitting')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.0 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Blind-B...


Loaded 80 Blind-B sessions
Cold start sessions: 40
Normal sessions:     40
Loading track metadata...
Loaded 47071 songs
Building track templates...
BM25 ready


Building song vectors...


Song vectors ready


Cross-encoder ready
Loading trained NLI model...


NLI model ready on cuda


100%|██████████| 80/80 [05:36<00:00,  4.21s/it]


Done! 80 predictions made
Expected 80 — Got 80
All 80 predictions ready — safe to submit
submission.zip ready!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import os

# Check common locations
locations = [
    '/content/blindb_predictions.ipynb',
    '/content/blindb_predictions (1).ipynb',
    '/content/drive/MyDrive/blindb_predictions.ipynb',
    '/content/drive/MyDrive/blindb_predictions (1).ipynb'
]

for loc in locations:
    if os.path.exists(loc):
        print(f'✅ Found at: {loc}')
    else:
        print(f'❌ Not at: {loc}')

❌ Not at: /content/blindb_predictions.ipynb
❌ Not at: /content/blindb_predictions (1).ipynb
❌ Not at: /content/drive/MyDrive/blindb_predictions.ipynb
❌ Not at: /content/drive/MyDrive/blindb_predictions (1).ipynb


In [5]:
import os

# Search entire drive for any .ipynb files
print('Searching for notebooks...')
for root, dirs, files in os.walk('/content'):
    for file in files:
        if file.endswith('.ipynb'):
            print(f'✅ Found: {os.path.join(root, file)}')

Searching for notebooks...
✅ Found: /content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/Copy of Untitled3.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/Untitled4.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/trailproject.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/trail2rec.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/usermetadata.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/track.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/Trackembeddings.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/userembeddings.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/music trail.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/Talkplaymusic.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/Untitled5.ipynb
✅ Found: /content/drive/MyDrive/Colab Notebooks/Un

In [6]:
import json, os
from google.colab import files

# Try the correct filename
path = '/content/drive/MyDrive/blinddb_predictions.ipynb'

if os.path.exists(path):
    print(f'✅ Found at: {path}')
else:
    print('❌ Still not found — run the search above')

❌ Still not found — run the search above
